In [1]:
import os, time, random
from collections import defaultdict

import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from tqdm import tqdm
from PIL import Image, ImageDraw, ImageFont, ImageFilter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split

SEED = 9999
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Пути
BASE_DIR   = "/kaggle/input/competitions/dl-lab-4-ocr"
TRAIN_DIR  = os.path.join(BASE_DIR, "train/train")
TEST_DIR   = os.path.join(BASE_DIR, "test/test")
TRAIN_CSV  = os.path.join(BASE_DIR, "train.csv")
TEST_CSV   = os.path.join(BASE_DIR, "test.csv")
SAMPLE_CSV = os.path.join(BASE_DIR, "sample_submission.csv")

# Размеры изображений
IMG_H, IMG_W = 32, 64

# CTC: 0=blank, 1-10 → цифры '0'-'9'
BLANK_IDX   = 0
VOCAB       = ['_'] + [str(d) for d in range(10)]
NUM_CLASSES = len(VOCAB)
CHAR2IDX    = {c: i for i, c in enumerate(VOCAB)}

# Найденные гиперпараметры
CHANNELS      = [64, 128, 256, 512]
HIDDEN        = 512
DROPOUT       = 0.3
WEIGHT_DECAY  = 1e-4

In [2]:
# ============================================================
# 1. LABEL ENCODING
# ============================================================

def price_to_seq(price: int) -> list[int]:
    return [CHAR2IDX[c] for c in str(price)]

def seq_to_price(seq: list[int]) -> str:
    return ''.join(VOCAB[i] for i in seq if i != BLANK_IDX)

def greedy_decode(log_probs: torch.Tensor) -> list[str]:
    indices = log_probs.argmax(dim=2)
    results = []
    for b in range(indices.size(1)):
        seq = indices[:, b].tolist()
        deduped = [seq[0]] + [seq[i] for i in range(1, len(seq))
                               if seq[i] != seq[i - 1]]
        chars = [VOCAB[i] for i in deduped if i != BLANK_IDX]
        results.append(''.join(chars))
    return results

In [3]:
# ============================================================
# 2. PREPROCESSING
# ============================================================

def preprocess_image(path: str) -> np.ndarray:
    raw     = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_COLOR)
    gray    = cv2.cvtColor(raw, cv2.COLOR_BGR2GRAY)
    eq      = cv2.equalizeHist(gray)
    resized = cv2.resize(eq, (IMG_W, IMG_H), interpolation=cv2.INTER_LINEAR)
    return resized

In [4]:
# ============================================================
# 3. AUGMENTATION PIPELINE
# ============================================================

_aug_train = A.Compose([
    A.ShiftScaleRotate(
        shift_limit=0.01, scale_limit=0.05, rotate_limit=3,
        border_mode=cv2.BORDER_REFLECT_101, p=0.3,
    ),
    A.OneOf([
        A.RandomBrightnessContrast(brightness_limit=0.08, contrast_limit=0.08, p=1.0),
        A.RandomGamma(gamma_limit=(92, 108), p=1.0),
    ], p=0.3),
    A.GaussNoise(std_range=(0.01, 0.03), p=0.15),
    A.OneOf([
        A.CoarseDropout(
            num_holes_range=(2, 8),
            hole_height_range=(2, 4),
            hole_width_range=(2, 4),
            fill=255, p=1.0,
        ),
        A.PixelDropout(dropout_prob=0.03, per_channel=False, drop_value=255, p=1.0),
    ], p=0.3),
    A.Normalize(mean=[0.5], std=[0.5]),
    ToTensorV2(),
])

_aug_val = A.Compose([
    A.Normalize(mean=[0.5], std=[0.5]),
    ToTensorV2(),
])

def apply_train_aug(img_gray: np.ndarray) -> torch.Tensor:
    hwc = img_gray[:, :, np.newaxis]
    return _aug_train(image=hwc)["image"]

def apply_val_aug(img_gray: np.ndarray) -> torch.Tensor:
    hwc = img_gray[:, :, np.newaxis]
    return _aug_val(image=hwc)["image"]

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [5]:
# ============================================================
# 4. DATA SPLIT (20% val, 80% train)
# ============================================================

train_df = pd.read_csv(TRAIN_CSV, sep='\t' if '\t' in open(TRAIN_CSV).read(1024) else ',')
train_df.columns = train_df.columns.str.strip()

class_counts = train_df["Price"].value_counts()
valid_cls    = class_counts[class_counts >= 2].index
df_valid     = train_df[train_df["Price"].isin(valid_cls)]
df_rare      = train_df[~train_df["Price"].isin(valid_cls)]

X_tv, X_val, y_tv, y_val = train_test_split(
    df_valid["Filename"], df_valid["Price"],
    test_size=0.2, random_state=42, stratify=df_valid["Price"],
)

X_train = pd.concat([X_tv, df_rare["Filename"]]).reset_index(drop=True)
y_train = pd.concat([y_tv, df_rare["Price"]]).reset_index(drop=True)
X_val   = X_val.reset_index(drop=True)
y_val   = y_val.reset_index(drop=True)

print(f"✓ Real data split: Train={len(X_train)}, Val={len(X_val)}")

✓ Real data split: Train=12040, Val=3010


In [6]:
# ============================================================
# 5. ГЕНЕРАЦИЯ СИНТЕТИЧЕСКИХ ДАННЫХ (с кешированием)
# ============================================================

font_paths = {
    'Rubik-Black': '/kaggle/input/datasets/tkachenko1van/rubik-fonts/Rubik-Black.ttf',
    'RubikDirt':   '/kaggle/input/datasets/tkachenko1van/rubik-fonts/RubikDirt-Regular.ttf',
}

def render_one_digit(digit: str, font_path: str, cell: int = 64) -> Image.Image:
    font_size = cell
    font = ImageFont.truetype(font_path, font_size)
    tmp  = Image.new('L', (cell * 2, cell * 2), 255)
    d    = ImageDraw.Draw(tmp)
    bb   = d.textbbox((0, 0), digit, font=font)
    glyph_h = bb[3] - bb[1]
    if glyph_h > 0:
        font_size = int(font_size * cell / glyph_h)
        font = ImageFont.truetype(font_path, font_size)
    img  = Image.new('L', (cell, cell), 255)
    draw = ImageDraw.Draw(img)
    bb   = draw.textbbox((0, 0), digit, font=font)
    gw   = bb[2] - bb[0]
    gh   = bb[3] - bb[1]
    x    = (cell - gw) // 2 - bb[0]
    y    = (cell - gh) // 2 - bb[1]
    draw.text((x, y), digit, fill=0, font=font)
    arr  = np.array(img)
    cols = np.any(arr < 200, axis=0)
    if not cols.any():
        return img
    c0, c1 = np.where(cols)[0][[0, -1]]
    return img.crop((c0, 0, c1 + 1, cell))

def make_number_mask(number: int) -> Image.Image:
    font_key  = random.choice(['Rubik-Black', 'RubikDirt'])
    font_path = font_paths[font_key]
    digits    = str(number)
    digit_imgs = []
    for ch in digits:
        di = render_one_digit(ch, font_path, cell=64)
        w, h    = di.size
        padded  = Image.new('L', (w + 2, h + 2), 255)
        padded.paste(di, (1, 1))
        digit_imgs.append(padded)
    total_w = sum(d.size[0] for d in digit_imgs)
    max_h   = max(d.size[1] for d in digit_imgs)
    result = Image.new('L', (total_w, max_h), 255)
    x_off  = 0
    for di in digit_imgs:
        y_off = (max_h - di.size[1]) // 2
        result.paste(di, (x_off, y_off))
        x_off += di.size[0]
    return result

def augment_mask(mask: Image.Image) -> Image.Image:
    scale = random.uniform(0.95, 1.05)
    nw    = max(1, int(mask.width  * scale))
    nh    = max(1, int(mask.height * scale))
    mask  = mask.resize((nw, nh), Image.LANCZOS)
    angle = random.uniform(-15.0, 15.0)
    mask  = mask.rotate(angle, expand=True, fillcolor=255, resample=Image.BICUBIC)
    return mask

def make_background(size: int = 128) -> Image.Image:
    h    = random.uniform(0.0, 1.0)
    rgb  = np.array([255.0, 255.0, 0.0])
    grey = np.mean(rgb)
    color = h * rgb + (1 - h) * grey
    bg    = np.ones((size, size, 3), dtype=np.float32) * color
    bg *= np.random.normal(1.0, 0.5, bg.shape)
    bg += np.random.normal(0.0, 100.0, bg.shape)
    sp = np.random.random((size, size))
    bg[sp < 0.05] = 0.0
    bg[sp > 0.95] = 255.0
    bg  = np.clip(bg, 0, 255).astype(np.uint8)
    img = Image.fromarray(bg)
    img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(4.0, 8.0)))
    return img

def generate_sample(number: int = None) -> tuple[Image.Image, int]:
    if number is None:
        n_dig  = random.choice([2, 3, 4])
        number = random.randint(10 ** (n_dig - 1), 10 ** n_dig - 1)
    mask = make_number_mask(number)
    mask = augment_mask(mask)
    mw, mh = mask.size
    bg       = make_background(128)
    bg_sized = bg.resize((mw, mh), Image.LANCZOS)
    mask_arr = np.array(mask).astype(np.float32) / 255.0
    bg_arr   = np.array(bg_sized).astype(np.float32)
    comp     = bg_arr * mask_arr[:, :, np.newaxis]
    comp += np.random.normal(0.0, 50.0, comp.shape)
    comp *= np.random.normal(1.0, 0.2, comp.shape)
    sp = np.random.random(comp.shape[:2])
    comp[sp < 0.025] = 0.0
    comp[sp > 0.975] = 255.0
    comp = np.clip(comp, 0, 255).astype(np.uint8)
    img  = Image.fromarray(comp)
    img = img.filter(ImageFilter.GaussianBlur(radius=random.uniform(1.0, 6.0)))
    img = img.filter(ImageFilter.GaussianBlur(radius=3))
    img = img.resize((random.randint(40, 60), random.randint(20, 30)), Image.LANCZOS)
    img = img.resize((IMG_W, IMG_H), Image.LANCZOS)
    return img, number

print("✓ Synthetic data generator ready")

# Генерация 100 000 синтетических изображений И КЕШИРОВАНИЕ В ПАМЯТЬ
N_SYNTHETIC = 100000
print(f"\n  Generating and caching {N_SYNTHETIC} synthetic images...")

synthetic_images = []
synthetic_prices = []

for _ in tqdm(range(N_SYNTHETIC), desc="Generating & caching", ncols=80):
    img, price = generate_sample()
    # PIL RGB → numpy gray (сразу препроцессинг)
    img_np = np.array(img.convert('L'))  # (32, 64) uint8
    synthetic_images.append(img_np)
    synthetic_prices.append(price)

# Кешируем в numpy массив (быстрый доступ по индексу)
synthetic_images = np.stack(synthetic_images, axis=0)  # (100000, 32, 64) uint8
synthetic_prices = np.array(synthetic_prices, dtype=np.int32)

print(f"  ✓ Generated & cached {N_SYNTHETIC} images")
print(f"    Memory: {synthetic_images.nbytes/1024/1024:.1f} MB")
print(f"    Shape: {synthetic_images.shape}")

# ============================================================
# Подготовка датасетов
# ============================================================

# Валидационный датасет (реальные данные) — кешируется при создании
print("\n  Preparing validation dataset (real data)...")
real_val_ds = PriceOCRDatasetFromFiles(X_val, y_val, TRAIN_DIR, apply_val_aug, cache=True)

# Синтетический датасет для предобучения (УЖЕ ЗАКЕШИРОВАН)
print("\n  Preparing synthetic train dataset...")
syn_train_ds = PriceOCRDataset(synthetic_images, synthetic_prices, apply_train_aug)
print(f"  ✓ Synthetic train: {len(syn_train_ds)} samples (CACHED)")

# Реальный датасет для дообучения (кешируется при создании)
print("\n  Preparing real train dataset...")
real_train_ds = PriceOCRDatasetFromFiles(X_train, y_train, TRAIN_DIR, 
                                          apply_train_aug, cache=True)

✓ Synthetic data generator ready

  Generating and caching 100000 synthetic images...


Generating & caching: 100%|█████████████| 100000/100000 [39:26<00:00, 42.26it/s]


  ✓ Generated & cached 100000 images
    Memory: 195.3 MB
    Shape: (100000, 32, 64)

  Preparing validation dataset (real data)...


NameError: name 'PriceOCRDatasetFromFiles' is not defined

In [ ]:
# ============================================================
# 6. DATASET & DATALOADER
# ============================================================

SMOOTH_ALPHA = 0.5

def _load_all_images(filenames, base_dir: str, desc: str) -> np.ndarray:
    imgs = []
    for fname in tqdm(filenames, desc=desc, ncols=80, leave=False):
        img = preprocess_image(os.path.join(base_dir, fname))
        imgs.append(img)
    return np.stack(imgs, axis=0)

class PriceOCRDataset(Dataset):
    """Датасет для работы с массивами изображений (синтетика или кешированные)"""
    def __init__(self, images, prices, augment_fn):
        self.images     = images
        self.prices     = np.array(prices, dtype=np.int32)
        self.augment_fn = augment_fn

    def __len__(self): return len(self.prices)

    def __getitem__(self, idx):
        img   = self.images[idx]  # (32, 64) uint8
        img_t = self.augment_fn(img)
        price = int(self.prices[idx])
        seq   = price_to_seq(price)
        return img_t, torch.tensor(seq, dtype=torch.long), price

class PriceOCRDatasetFromFiles(Dataset):
    """Датасет для загрузки из файлов с кешированием"""
    def __init__(self, filenames, prices, base_dir, augment_fn, cache=True):
        fnames  = np.array(filenames)
        prices_ = np.array(prices, dtype=np.int32)
        self.filenames  = fnames
        self.prices     = prices_
        self.base_dir   = base_dir
        self.augment_fn = augment_fn
        self.cache      = cache
        if cache:
            self.images = _load_all_images(fnames, base_dir, f"Caching {len(fnames)} imgs")
            print(f"  ✓ {len(fnames)} imgs cached ({self.images.nbytes/1024/1024:.1f} MB)")

    def __len__(self): return len(self.filenames)

    def __getitem__(self, idx):
        img   = self.images[idx] if self.cache else preprocess_image(
                    os.path.join(self.base_dir, self.filenames[idx]))
        img_t = self.augment_fn(img)
        price = int(self.prices[idx])
        seq   = price_to_seq(price)
        return img_t, torch.tensor(seq, dtype=torch.long), price

def collate_fn(batch):
    imgs, seqs, prices = zip(*batch)
    imgs    = torch.stack(imgs)
    lengths = torch.tensor([len(s) for s in seqs], dtype=torch.long)
    max_len = int(lengths.max())
    padded  = torch.zeros(len(seqs), max_len, dtype=torch.long)
    for i, s in enumerate(seqs):
        padded[i, :len(s)] = s
    prices  = torch.tensor(prices, dtype=torch.long)
    return imgs, padded, lengths, prices

def make_weighted_sampler(prices):
    freq_map = pd.Series(prices).value_counts().to_dict()
    w = np.array([(1.0/freq_map[p])**SMOOTH_ALPHA for p in prices], dtype=np.float32)
    w /= w.sum()
    return WeightedRandomSampler(w, num_samples=len(w), replacement=True)

In [ ]:
# ============================================================
# Подготовка датасетов
# ============================================================

# Валидационный датасет (реальные данные) — используется везде
print("\n  Preparing validation dataset (real data)...")
real_val_ds = PriceOCRDatasetFromFiles(X_val, y_val, TRAIN_DIR, apply_val_aug, cache=True)

# Синтетический датасет для предобучения (train)
print("\n  Preparing synthetic train dataset...")
syn_train_ds = PriceOCRDataset(synthetic_images, synthetic_prices, apply_train_aug)
print(f"  ✓ Synthetic train: {len(syn_train_ds)} samples")

# Реальный датасет для дообучения (train)
print("\n  Preparing real train dataset...")
real_train_ds = PriceOCRDatasetFromFiles(X_train, y_train, TRAIN_DIR, 
                                          apply_train_aug, cache=True)

In [ ]:
# ============================================================
# ПРОВЕРКА АУГМЕНТАЦИЙ НА СИНТЕТИКЕ
# ============================================================

print("\n" + "=" * 60)
print("  ПРОВЕРКА АУГМЕНТАЦИЙ")
print("=" * 60)

# Берём одно синтетическое изображение
idx = 0
img_clean = synthetic_images[idx]  # (32, 64) uint8

print(f"Чистое изображение: shape={img_clean.shape}, dtype={img_clean.dtype}")
print(f"Цена: {synthetic_prices[idx]}")

# Применяем аугментации 5 раз
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
fig.suptitle(f"Synthetic image augmentations (price={synthetic_prices[idx]})", fontsize=12)

# Оригинал
axes[0].imshow(img_clean, cmap='gray', aspect='auto')
axes[0].set_title("Original (clean)")
axes[0].axis('off')

# 5 аугментаций
for i in range(5):
    img_aug = apply_train_aug(img_clean)  # (1, 32, 64) tensor
    img_np = img_aug.squeeze(0).numpy()   # (32, 64) float32 [-1, 1]
    axes[i+1].imshow(img_np, cmap='gray', vmin=-1, vmax=1, aspect='auto')
    axes[i+1].set_title(f"Aug {i+1}")
    axes[i+1].axis('off')

plt.tight_layout()
plt.savefig("synthetic_augmentation_check.png", dpi=100, bbox_inches='tight')
plt.show()
print("✓ Saved: synthetic_augmentation_check.png")

print("\nПроверка через DataLoader:")
# Создаём временный лоадер
temp_loader = DataLoader(syn_train_ds, batch_size=4, shuffle=True, collate_fn=collate_fn)
imgs, _, _, prices = next(iter(temp_loader))
print(f"  Batch shape: {imgs.shape}")  # (4, 1, 32, 64)
print(f"  Dtype: {imgs.dtype}")        # torch.float32
print(f"  Range: [{imgs.min():.2f}, {imgs.max():.2f}]")  # [-1, 1] после Normalize
print(f"  Prices: {prices.tolist()}")

print("=" * 60)
print("✓ Аугментации применяются корректно!")
print("=" * 60)

In [ ]:
# ============================================================
# 7. MODEL: STN
# ============================================================

class STN(nn.Module):
    def __init__(self):
        super().__init__()
        self.loc = nn.Sequential(
            nn.Conv2d(1,  32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2, 2),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.fc_loc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Linear(32, 6),
        )
        self.fc_loc[-1].weight.data.zero_()
        self.fc_loc[-1].bias.data.copy_(
            torch.tensor([1, 0, 0, 0, 1, 0], dtype=torch.float32)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        theta_raw = self.fc_loc(self.loc(x))
        theta_raw = theta_raw.view(-1, 2, 3)
        scale_x = 0.2  * torch.tanh(theta_raw[:, 0, 0]) + 1.0
        scale_y = 0.2  * torch.tanh(theta_raw[:, 1, 1]) + 1.0
        shear_x = 0.268 * torch.tanh(theta_raw[:, 0, 1])
        shear_y = 0.268 * torch.tanh(theta_raw[:, 1, 0])
        tx      = theta_raw[:, 0, 2]
        ty      = theta_raw[:, 1, 2]
        theta = torch.stack([
            torch.stack([scale_x, shear_x, tx], dim=1),
            torch.stack([shear_y, scale_y, ty], dim=1),
        ], dim=1)
        grid = F.affine_grid(theta, x.size(), align_corners=False)
        return F.grid_sample(x, grid, align_corners=False)

In [ ]:
# ============================================================
# 8. MODEL: CRNN
# ============================================================

class CRNN(nn.Module):
    def __init__(self, channels: list[int], hidden: int, dropout: float = 0.3,
                 num_classes: int = NUM_CLASSES):
        super().__init__()
        c1, c2, c3, c4 = channels
        self.cnn = nn.Sequential(
            nn.Conv2d(1,  c1, kernel_size=3, padding=1),
            nn.BatchNorm2d(c1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2)),
            nn.Conv2d(c1, c2, kernel_size=3, padding=1),
            nn.BatchNorm2d(c2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1)),
            nn.Conv2d(c2, c3, kernel_size=3, padding=1),
            nn.BatchNorm2d(c3),
            nn.ReLU(inplace=True),
            nn.Conv2d(c3, c3, kernel_size=3, padding=1),
            nn.BatchNorm2d(c3),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 1)),
            nn.Conv2d(c3, c4, kernel_size=3, padding=1),
            nn.BatchNorm2d(c4),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=(4, 1), stride=(4, 1)),
        )
        self.rnn = nn.LSTM(
            input_size=c4, hidden_size=hidden, num_layers=2,
            batch_first=False, bidirectional=True, dropout=dropout,
        )
        self.fc = nn.Linear(hidden * 2, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        feat = self.cnn(x)
        feat = feat.squeeze(2)
        feat = feat.permute(2, 0, 1)
        out, _ = self.rnn(feat)
        logits = self.fc(out)
        return F.log_softmax(logits, dim=2)

In [ ]:
# ============================================================
# 9. ПОЛНАЯ МОДЕЛЬ: STN + CRNN
# ============================================================

class STN_CRNN(nn.Module):
    def __init__(self, channels: list[int], hidden: int, dropout: float = 0.3):
        super().__init__()
        self.stn  = STN()
        self.crnn = CRNN(channels, hidden, dropout)

    def forward(self, x: torch.Tensor, use_stn: bool = True) -> torch.Tensor:
        if use_stn:
            x = self.stn(x)
        return self.crnn(x)

In [ ]:
# ============================================================
# 10. МЕТРИКИ
# ============================================================

def compute_metrics(log_probs: torch.Tensor, prices: torch.Tensor):
    preds = greedy_decode(log_probs)
    trues = [str(int(p)) for p in prices]
    exact, total_chars, wrong_chars = 0, 0, 0
    for pred, true in zip(preds, trues):
        exact += int(pred == true)
        total_chars += len(true)
        wrong_chars += _levenshtein(pred, true)
    n = len(preds)
    cer = wrong_chars / max(total_chars, 1)
    return exact / n, cer

def _levenshtein(s1: str, s2: str) -> int:
    if s1 == s2: return 0
    m, n = len(s1), len(s2)
    dp = list(range(n + 1))
    for i in range(1, m + 1):
        prev, dp[0] = dp[0], i
        for j in range(1, n + 1):
            prev, dp[j] = dp[j], (prev if s1[i-1]==s2[j-1]
                                   else 1 + min(prev, dp[j], dp[j-1]))
    return dp[n]

In [ ]:
# ============================================================
# 11. ФУНКЦИИ ОБУЧЕНИЯ
# ============================================================

def make_scheduler(optimizer, warmup_epochs, total_epochs, base_lr, min_lr_factor=1e-5):
    def warmup_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        return 1.0
    scheduler_w = torch.optim.lr_scheduler.LambdaLR(optimizer, warmup_lambda)
    scheduler_c = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=max(total_epochs - warmup_epochs, 1),
        eta_min=base_lr * min_lr_factor,
    )
    return torch.optim.lr_scheduler.SequentialLR(
        optimizer, [scheduler_w, scheduler_c], milestones=[warmup_epochs]
    )

ctc_loss_fn = nn.CTCLoss(blank=BLANK_IDX, reduction='mean', zero_infinity=True)

def run_epoch(model, loader, optimizer, use_stn, is_train,
              freeze_crnn=False, freeze_stn=False):
    model.train() if is_train else model.eval()
    if freeze_crnn:
        for p in model.crnn.parameters(): p.requires_grad_(False)
    else:
        for p in model.crnn.parameters(): p.requires_grad_(True)
    if freeze_stn:
        for p in model.stn.parameters(): p.requires_grad_(False)
    else:
        for p in model.stn.parameters(): p.requires_grad_(True)

    total_loss, total_em, total_cer, n_batches = 0., 0., 0., 0
    ctx = torch.no_grad() if not is_train else torch.enable_grad()
    
    with ctx:
        for imgs, seqs, lengths, prices in loader:
            imgs    = imgs.to(device, non_blocking=True)
            seqs    = seqs.to(device, non_blocking=True)
            lengths = lengths.to(device, non_blocking=True)
            prices  = prices.to(device, non_blocking=True)

            log_probs = model(imgs, use_stn=use_stn)
            T, B, _   = log_probs.shape
            input_lengths = torch.full((B,), T, dtype=torch.long, device=device)
            loss = ctc_loss_fn(log_probs, seqs, input_lengths, lengths)

            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()

            em, cer = compute_metrics(log_probs.detach(), prices.cpu())
            total_loss += loss.item()
            total_em   += em
            total_cer  += cer
            n_batches  += 1

    return total_loss/n_batches, total_em/n_batches, total_cer/n_batches

In [ ]:
# ============================================================
# 12. ПРЕДОБУЧЕНИЕ НА СИНТЕТИЧЕСКИХ ДАННЫХ (3 этапа)
#     Train: 100k синтетики
#     Val: 20% реальных данных
# ============================================================

print("\n" + "=" * 60)
print("  ПРЕДОБУЧЕНИЕ НА СИНТЕТИЧЕСКИХ ДАННЫХ")
print("  Train: 100k synthetic | Val: real 20%")
print("=" * 60)

BATCH_SIZE_PRETRAIN = 256
PRE_LR = 1e-3

def make_loaders_pretrain(batch_size):
    sampler  = make_weighted_sampler(syn_train_ds.prices.tolist())
    train_ld = DataLoader(syn_train_ds, batch_size=batch_size, sampler=sampler,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
    val_ld   = DataLoader(real_val_ds, batch_size=batch_size, shuffle=False,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
    return train_ld, val_ld

pretrain_train_ld, pretrain_val_ld = make_loaders_pretrain(BATCH_SIZE_PRETRAIN)

model = STN_CRNN(CHANNELS, HIDDEN, DROPOUT).to(device)

def _save_best(model) -> dict:
    return {k: v.cpu().clone() for k, v in model.state_dict().items()}

def _load_best(model, best_state: dict):
    model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

# ── ЭТАП 0: Предобучение CRNN без STN ──
STAGE0_EPOCHS = 20
STAGE0_WARMUP = 3

print(f"\n┌─ Stage 0: CRNN pre-train — {STAGE0_EPOCHS} epochs")

opt0 = torch.optim.AdamW(model.crnn.parameters(), lr=PRE_LR, weight_decay=WEIGHT_DECAY)
sch0 = make_scheduler(opt0, STAGE0_WARMUP, STAGE0_EPOCHS, PRE_LR)

best_vem_s0 = 0.0
best_state_s0 = _save_best(model)

for ep in range(1, STAGE0_EPOCHS + 1):
    t0 = time.time()
    tl, tem, tcer = run_epoch(model, pretrain_train_ld, opt0, use_stn=False,
                              is_train=True, freeze_stn=True)
    vl, vem, vcer = run_epoch(model, pretrain_val_ld, opt0, use_stn=False,
                              is_train=False, freeze_stn=True)
    sch0.step()
    elapsed = time.time() - t0
    print(f"[S0] ep {ep:>2}/{STAGE0_EPOCHS} │ train loss={tl:.4f} em={tem:.3f} │ "
          f"val loss={vl:.4f} em={vem:.3f} │ {elapsed:.1f}s")
    
    if vem > best_vem_s0:
        best_vem_s0 = vem
        best_state_s0 = _save_best(model)

_load_best(model, best_state_s0)
print(f"✓ Stage 0 best val EM = {best_vem_s0:.4f}")

# ── ЭТАП 1: Обучение STN, заморозка CRNN ──
STAGE1_EPOCHS = 20
STAGE1_WARMUP = 3

print(f"\n├─ Stage 1: STN train — {STAGE1_EPOCHS} epochs")

opt1 = torch.optim.AdamW(model.stn.parameters(), lr=PRE_LR, weight_decay=WEIGHT_DECAY)
sch1 = make_scheduler(opt1, STAGE1_WARMUP, STAGE1_EPOCHS, PRE_LR)

best_vem_s1 = 0.0
best_state_s1 = _save_best(model)

for ep in range(1, STAGE1_EPOCHS + 1):
    t0 = time.time()
    tl, tem, tcer = run_epoch(model, pretrain_train_ld, opt1, use_stn=True,
                              is_train=True, freeze_crnn=True)
    vl, vem, vcer = run_epoch(model, pretrain_val_ld, opt1, use_stn=True,
                              is_train=False, freeze_crnn=True)
    sch1.step()
    elapsed = time.time() - t0
    print(f"[S1] ep {ep:>2}/{STAGE1_EPOCHS} │ train loss={tl:.4f} em={tem:.3f} │ "
          f"val loss={vl:.4f} em={vem:.3f} │ {elapsed:.1f}s")
    
    if vem > best_vem_s1:
        best_vem_s1 = vem
        best_state_s1 = _save_best(model)

_load_best(model, best_state_s1)
print(f"✓ Stage 1 best val EM = {best_vem_s1:.4f}")

# ── ЭТАП 2: Совместное обучение STN + CRNN ──
STAGE2_EPOCHS = 40

print(f"\n└─ Stage 2: Joint fine-tune — {STAGE2_EPOCHS} epochs")

opt2 = torch.optim.AdamW(model.parameters(), lr=PRE_LR*0.1, weight_decay=WEIGHT_DECAY)
sch2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt2, mode='max', factor=0.2, patience=7, min_lr=PRE_LR*1e-4
)

best_vem_s2 = 0.0
best_state_s2 = _save_best(model)

for ep in range(1, STAGE2_EPOCHS + 1):
    t0 = time.time()
    tl, tem, tcer = run_epoch(model, pretrain_train_ld, opt2, use_stn=True, is_train=True)
    vl, vem, vcer = run_epoch(model, pretrain_val_ld, opt2, use_stn=True, is_train=False)
    sch2.step(vem)
    elapsed = time.time() - t0
    print(f"[S2] ep {ep:>2}/{STAGE2_EPOCHS} │ train loss={tl:.4f} em={tem:.3f} │ "
          f"val loss={vl:.4f} em={vem:.3f} │ {elapsed:.1f}s")
    
    if vem > best_vem_s2:
        best_vem_s2 = vem
        best_state_s2 = _save_best(model)

_load_best(model, best_state_s2)
print(f"\n✓ Pretraining complete!")
print(f"  Stage 0 best val EM: {best_vem_s0:.4f}")
print(f"  Stage 1 best val EM: {best_vem_s1:.4f}")
print(f"  Stage 2 best val EM: {best_vem_s2:.4f}")

torch.save(model.state_dict(), "pretrained_synthetic.pt")

In [ ]:
# ============================================================
# 13. ДООБУЧЕНИЕ НА РЕАЛЬНЫХ ДАННЫХ
#     Train: 80% реальных данных
#     Val: те же 20% реальных данных
# ============================================================

print("\n" + "=" * 60)
print("  ДООБУЧЕНИЕ НА РЕАЛЬНЫХ ДАННЫХ")
print("  Train: real 80% | Val: real 20%")
print("=" * 60)

BATCH_SIZE_FINETUNE = 256
FINE_LR = 1e-5
FINE_EPOCHS = 80

def make_loaders_finetune(batch_size):
    sampler  = make_weighted_sampler(real_train_ds.prices.tolist())
    train_ld = DataLoader(real_train_ds, batch_size=batch_size, sampler=sampler,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
    val_ld   = DataLoader(real_val_ds, batch_size=batch_size, shuffle=False,
                          collate_fn=collate_fn, num_workers=2, pin_memory=True)
    return train_ld, val_ld

finetune_train_ld, finetune_val_ld = make_loaders_finetune(BATCH_SIZE_FINETUNE)

opt_fine = torch.optim.AdamW(model.parameters(), lr=FINE_LR, weight_decay=WEIGHT_DECAY)
sch_fine = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt_fine, mode='max', factor=0.2, patience=7, min_lr=FINE_LR*1e-4
)

best_vem_fine = 0.0
best_state_fine = _save_best(model)

print(f"\nFinetuning for {FINE_EPOCHS} epochs (lr={FINE_LR})")

for ep in range(1, FINE_EPOCHS + 1):
    t0 = time.time()
    tl, tem, tcer = run_epoch(model, finetune_train_ld, opt_fine, use_stn=True, is_train=True)
    vl, vem, vcer = run_epoch(model, finetune_val_ld, opt_fine, use_stn=True, is_train=False)
    sch_fine.step(vem)
    elapsed = time.time() - t0
    
    print(f"[FT] ep {ep:>2}/{FINE_EPOCHS} │ train loss={tl:.4f} em={tem:.3f} │ "
          f"val loss={vl:.4f} em={vem:.3f} │ {elapsed:.1f}s")
    
    if vem > best_vem_fine:
        best_vem_fine = vem
        best_state_fine = _save_best(model)

_load_best(model, best_state_fine)
print(f"\n✓ Finetuning complete! Best val EM = {best_vem_fine:.4f}")

torch.save(model.state_dict(), "final_model.pt")

In [ ]:
# ============================================================
# 14. ИНФЕРЕНС ТЕСТОВОГО ДАТАСЕТА
# ============================================================

print("\n" + "=" * 60)
print("  INFERENCE ON TEST SET")
print("=" * 60)

test_df = pd.read_csv(TEST_CSV)
test_df.columns = test_df.columns.str.strip()
test_filenames = test_df['Filename'].values

print(f"Test samples: {len(test_filenames)}")

# Препроцессинг и инференс
model.eval()
predictions = []

with torch.no_grad():
    for fname in tqdm(test_filenames, desc="Inference", ncols=80):
        path = os.path.join(TEST_DIR, fname)
        img  = preprocess_image(path)  # (32, 64) uint8
        img_t = apply_val_aug(img)     # (1, 32, 64) float32
        img_t = img_t.unsqueeze(0).to(device)  # (1, 1, 32, 64)
        
        log_probs = model(img_t, use_stn=True)  # (T, 1, C)
        pred_str  = greedy_decode(log_probs)[0]
        
        # Конвертируем в int (если пусто — ставим 0)
        try:
            price = int(pred_str) if pred_str else 0
        except:
            price = 0
        
        predictions.append(price)

# Создание submission
submission = pd.DataFrame({
    'Filename': test_filenames,
    'Price': predictions
})

submission.to_csv('submission.csv', index=False)
print("✓ Saved: submission.csv")
print(f"\nSample predictions:")
print(submission.head(10))

print("\n" + "=" * 60)
print("  ✓ PIPELINE COMPLETE")
print("=" * 60)
print(f"\nFinal validation ExactMatch: {best_vem_fine:.4f}")
print(f"Files saved:")
print(f"  - pretrained_synthetic.pt")
print(f"  - final_model.pt")
print(f"  - submission.csv")